# Assignment 4
## Description
In this assignment you must read in a file of metropolitan regions and associated sports teams from [assets/wikipedia_data.html](assets/wikipedia_data.html) and answer some questions about each metropolitan region. Each of these regions may have one or more teams from the "Big 4": NFL (football, in [assets/nfl.csv](assets/nfl.csv)), MLB (baseball, in [assets/mlb.csv](assets/mlb.csv)), NBA (basketball, in [assets/nba.csv](assets/nba.csv) or NHL (hockey, in [assets/nhl.csv](assets/nhl.csv)). Please keep in mind that all questions are from the perspective of the metropolitan region, and that this file is the "source of authority" for the location of a given sports team. Thus teams which are commonly known by a different area (e.g. "Oakland Raiders") need to be mapped into the metropolitan region given (e.g. San Francisco Bay Area). This will require some human data understanding outside of the data you've been given (e.g. you will have to hand-code some names, and might need to google to find out where teams are)!

For each sport I would like you to answer the question: **what is the win/loss ratio's correlation with the population of the city it is in?** Win/Loss ratio refers to the number of wins over the number of wins plus the number of losses. Remember that to calculate the correlation with [`pearsonr`]. You should only use data **from year 2018** for your analysis -- this is important!

## Question 1 - 4
Calculate the win/loss ratio's correlation with the population of the city it is in for the **Each Sport** using **2018** data.

### Read, Wrangle, and Merge Data

In [1]:
## SETUP
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

data_year = 2018
sports = ['nhl', 'nba', 'mlb', 'nfl']

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
## READ-IN FILES

## NHL
nhl_df=pd.read_csv("/kaggle/input/big4-sports-league-data/nhl.csv")
nhl_df=nhl_df[(nhl_df.year == data_year) & (nhl_df.team.apply(lambda a: 'Division' not in a))][['team', 'W', 'L', 'year', 'League']].replace(to_replace='\*', value='', regex=True)

## NBA
nba_df=pd.read_csv("/kaggle/input/big4-sports-league-data/nba.csv")
nba_df=nba_df[(nba_df.year == data_year) & (nba_df.team.apply(lambda a: 'Division' not in a))][['team', 'W', 'L', 'year', 'League']].replace(to_replace='\*?\u00AD?\s?\(\d+\)', value='', regex=True)

## MLB
mlb_df=pd.read_csv("/kaggle/input/big4-sports-league-data/mlb.csv")
mlb_df=mlb_df[(mlb_df.year == data_year) & (mlb_df.team.apply(lambda a: 'Division' not in a))][['team', 'W', 'L', 'year', 'League']].replace(to_replace='\*?\u00AD?\s?\(\d+\)', value='', regex=True)

## NFL
nfl_df=pd.read_csv("/kaggle/input/big4-sports-league-data/nfl.csv")
nfl_df=nfl_df[(nfl_df.year == data_year) & (nfl_df.team.apply(lambda a: ('AFC' not in a) & ('NFC' not in a)))][['team', 'W', 'L', 'year', 'League']].replace(to_replace='\W$', value='', regex=True)

## WIKIPEDIA HTML
cities=pd.read_html("/kaggle/input/big4-sports-league-data/wikipedia_data.html")[1]
cities=(
    cities.iloc[:-1,[0,3,5,6,7,8]]
    .replace(to_replace='\—', value=np.nan, regex=True)
    .replace(to_replace='\[.*\]', value='', regex=True).replace(to_replace='', value=np.nan, regex=True)
    .rename(columns={'Population (2016 est.)[8]':'Population'}))

In [3]:
nhl_df.head()

,team,W,L,year,League
1,Tampa Bay Lightning,54,23,2018,NHL
2,Boston Bruins,50,20,2018,NHL
3,Toronto Maple Leafs,49,26,2018,NHL
4,Florida Panthers,44,30,2018,NHL
5,Detroit Red Wings,30,39,2018,NHL


In [4]:
cities.head()

,Metropolitan area,Population,NFL,MLB,NBA,NHL
0,New York City,20153634,Giants Jets,Yankees Mets,Knicks Nets,Rangers Islanders Devils
1,Los Angeles,13310447,Rams Chargers,Dodgers Angels,Lakers Clippers,Kings Ducks
2,San Francisco Bay Area,6657982,49ers Raiders,Giants Athletics,Warriors,Sharks
3,Chicago,9512999,Bears,Cubs White Sox,Bulls,Blackhawks
4,Dallas–Fort Worth,7233323,Cowboys,Rangers,Mavericks,Stars


In [5]:
## DEFINE FUNCTION TO PARSE NAMES FROM EACH LEAGUE FILE
twoWord_Metro = []
twoWord_Team = []

cityName_dict = {
'cityname_check_nhl': ['Tampa Bay', 'New York', 'New Jersey', 'St. Louis', 'San Jose', 'Los Angeles'],
'cityname_check_nba': ['Golden State', 'New York', 'New Orleans', 'Oklahoma City', 'San Antonio', 'Los Angeles'],
'cityname_check_mlb': ['New York', 'Kansas City', 'St. Louis', 'San Francisco', 'Los Angeles', 'San Diego', 'Tampa Bay'],
'cityname_check_nfl': ['New York', 'Kansas City', 'New Orleans', 'San Francisco', 'Los Angeles', 'New England', 'Tampa Bay', 'Green Bay']}

teamName_dict = {
'team_check_nhl': ['Toronto', 'Detroit', 'Columbus', 'Vegas'],
'team_check_nba': ['Portland'],
'team_check_mlb': ['Boston', 'Chicago', 'Toronto'],
'team_check_nfl': []}

for key in cityName_dict:
    for item in cityName_dict[key]:
        if item not in twoWord_Metro:
            twoWord_Metro.append(item)

for key in teamName_dict:
    for item in teamName_dict[key]:
        if item not in twoWord_Team:
            twoWord_Team.append(item)

def findCity(x):
    temp_string = x.split(' ')
    errorList = []

    if any(a in x for a in twoWord_Metro) and len(temp_string)==3:
        return (temp_string[0]+' '+temp_string[1], temp_string[2])
    elif any(b in x for b in twoWord_Team) and len(temp_string)==3:
        return (temp_string[0], temp_string[1]+' '+temp_string[2])
    else:
        return (temp_string[0], temp_string[1])
    
nhl_df[['Metro', 'Team Name']] = nhl_df['team'].apply(findCity).apply(pd.Series)
nba_df[['Metro', 'Team Name']] = nba_df['team'].apply(findCity).apply(pd.Series)
mlb_df[['Metro', 'Team Name']] = mlb_df['team'].apply(findCity).apply(pd.Series)
nfl_df[['Metro', 'Team Name']] = nfl_df['team'].apply(findCity).apply(pd.Series)

nhl_df.head()

,team,W,L,year,League,Metro,Team Name
1,Tampa Bay Lightning,54,23,2018,NHL,Tampa Bay,Lightning
2,Boston Bruins,50,20,2018,NHL,Boston,Bruins
3,Toronto Maple Leafs,49,26,2018,NHL,Toronto,Maple Leafs
4,Florida Panthers,44,30,2018,NHL,Florida,Panthers
5,Detroit Red Wings,30,39,2018,NHL,Detroit,Red Wings


In [6]:
## CALCULATE WIN-LOSS RATIO
nhl_df[['W','L']] = nhl_df[['W','L']].astype('int64')
nba_df[['W','L']] = nba_df[['W','L']].astype('int64')
mlb_df[['W','L']] = mlb_df[['W','L']].astype('int64')
nfl_df[['W','L']] = nfl_df[['W','L']].astype('int64')

nhl_df['WL_Ratio'] = nhl_df['W']/(nhl_df['W']+nhl_df['L'])
nba_df['WL_Ratio'] = nba_df['W']/(nba_df['W']+nba_df['L'])
mlb_df['WL_Ratio'] = mlb_df['W']/(mlb_df['W']+mlb_df['L'])
nfl_df['WL_Ratio'] = nfl_df['W']/(nfl_df['W']+nfl_df['L']) 

nhl_df.head()

,team,W,L,year,League,Metro,Team Name,WL_Ratio
1,Tampa Bay Lightning,54,23,2018,NHL,Tampa Bay,Lightning,0.701299
2,Boston Bruins,50,20,2018,NHL,Boston,Bruins,0.714286
3,Toronto Maple Leafs,49,26,2018,NHL,Toronto,Maple Leafs,0.653333
4,Florida Panthers,44,30,2018,NHL,Florida,Panthers,0.594595
5,Detroit Red Wings,30,39,2018,NHL,Detroit,Red Wings,0.434783


In [7]:
## EXPAND CITIES DATAFRAME TO CREATE ONE ROW PER TEAM
cities_explode_nhl = cities[['Metropolitan area','NHL']][pd.isna(cities.NHL)==False]
cities_explode_nhl['NHL'] = cities_explode_nhl['NHL'].str.split()
cities_explode_nhl = cities_explode_nhl.explode('NHL')
cities_explode_nhl = cities_explode_nhl.replace({'NHL': {'Golden':'Golden Knights','Red': 'Red Wings', 'Maple': 'Maple Leafs', 'Blue': 'Blue Jackets'}},).reset_index(drop=True).drop([16,18,27,33], axis=0)
cities_explode_nhl.rename({'NHL': 'Team Name'}, axis='columns', inplace=True)

cities_explode_nba = cities[['Metropolitan area', 'NBA']][pd.isna(cities.NBA)==False]
cities_explode_nba['NBA'] = cities_explode_nba['NBA'].str.split()
cities_explode_nba = cities_explode_nba.explode('NBA')
cities_explode_nba = cities_explode_nba.replace({'NBA': {'Trail':'Trail Blazers'}}).reset_index(drop=True).drop([25], axis=0)
cities_explode_nba.rename({'NBA': 'Team Name'}, axis='columns', inplace=True)

cities_explode_mlb = cities[['Metropolitan area','MLB']][pd.isna(cities.MLB)==False]
cities_explode_mlb['MLB'] = cities_explode_mlb['MLB'].str.split()
cities_explode_mlb = cities_explode_mlb.explode('MLB')
cities_explode_mlb.reset_index(drop=True, inplace=True)
cities_explode_mlb = cities_explode_mlb.replace({'MLB': {'White':'White Sox','Red':'Red Sox', 'Blue':'Blue Jays',}}).drop([8, 13, 20], axis=0).reset_index(drop=True)
cities_explode_mlb.rename({'MLB': 'Team Name'}, axis='columns', inplace=True)

cities_explode_nfl = cities[['Metropolitan area', 'NFL']][pd.isna(cities.NFL)==False]
cities_explode_nfl['NFL'] = cities_explode_nfl['NFL'].str.split()
cities_explode_nfl = cities_explode_nfl.explode('NFL')
cities_explode_nfl.reset_index(drop=True, inplace=True)
cities_explode_nfl.rename({'NFL': 'Team Name'}, axis='columns', inplace=True)

cities_explode_nhl.head()

,Metropolitan area,Team Name
0,New York City,Rangers
1,New York City,Islanders
2,New York City,Devils
3,Los Angeles,Kings
4,Los Angeles,Ducks


In [8]:
## MERGE EXPANDED CITIES DF WITH SPORT DATAFRAMES
merged_df_nhl = nhl_df.merge(cities_explode_nhl, on='Team Name')
merged_df_nba = nba_df.merge(cities_explode_nba, on='Team Name')
merged_df_mlb = mlb_df.merge(cities_explode_mlb, on='Team Name')
merged_df_nfl = nfl_df.merge(cities_explode_nfl, on='Team Name')

merged_df_nhl.head()

,team,W,L,year,League,Metro,Team Name,WL_Ratio,Metropolitan area
0,Tampa Bay Lightning,54,23,2018,NHL,Tampa Bay,Lightning,0.701299,Tampa Bay Area
1,Boston Bruins,50,20,2018,NHL,Boston,Bruins,0.714286,Boston
2,Toronto Maple Leafs,49,26,2018,NHL,Toronto,Maple Leafs,0.653333,Toronto
3,Florida Panthers,44,30,2018,NHL,Florida,Panthers,0.594595,Miami–Fort Lauderdale
4,Detroit Red Wings,30,39,2018,NHL,Detroit,Red Wings,0.434783,Detroit


In [9]:
## CALCULATE AVERAGE WIN-LOSE RATIOS FOR AREAS WITH MULTIPLE TEAMS
avgWL_nhl = merged_df_nhl[['Metropolitan area', 'WL_Ratio']].groupby(['Metropolitan area']).transform(np.nanmean)
avgWL_nhl.rename({'WL_Ratio': 'AvgWL_Ratio_nhl'}, axis=1, inplace=True)

avgWL_nba = merged_df_nba[['Metropolitan area', 'WL_Ratio']].groupby(['Metropolitan area']).transform(np.nanmean)
avgWL_nba.rename({'WL_Ratio': 'AvgWL_Ratio_nba'}, axis=1, inplace=True)

avgWL_mlb = merged_df_mlb[['Metropolitan area', 'WL_Ratio']].groupby(['Metropolitan area']).transform(np.nanmean)
avgWL_mlb.rename({'WL_Ratio': 'AvgWL_Ratio_mlb'}, axis=1, inplace=True)

avgWL_nfl = merged_df_nfl[['Metropolitan area', 'WL_Ratio']].groupby(['Metropolitan area']).transform(np.nanmean)
avgWL_nfl.rename({'WL_Ratio': 'AvgWL_Ratio_nfl'}, axis=1, inplace=True)

In [10]:
## COLLAPSE ROWS INTO METROPOLITAN AREAS DICTATED BY WIKIPEDIA SOURCE
final_df_nhl = merged_df_nhl[['League', 'year', 'Metropolitan area']].merge(avgWL_nhl, right_index=True, left_index=True).drop_duplicates()
final_df_nba = merged_df_nba[['League', 'year', 'Metropolitan area']].merge(avgWL_nba, right_index=True, left_index=True).drop_duplicates()
final_df_mlb = merged_df_mlb[['League', 'year', 'Metropolitan area']].merge(avgWL_mlb, right_index=True, left_index=True).drop_duplicates()
final_df_nfl = merged_df_nfl[['League', 'year', 'Metropolitan area']].merge(avgWL_nfl, right_index=True, left_index=True).drop_duplicates()

final_df_nhl.head(10)

,League,year,Metropolitan area,AvgWL_Ratio_nhl
0,NHL,2018,Tampa Bay Area,0.701299
1,NHL,2018,Boston,0.714286
2,NHL,2018,Toronto,0.653333
3,NHL,2018,Miami–Fort Lauderdale,0.594595
4,NHL,2018,Detroit,0.434783
5,NHL,2018,Montreal,0.420290
6,NHL,2018,Ottawa,0.394366
7,NHL,2018,Buffalo,0.357143
8,NHL,2018,"Washington, D.C.",0.653333
9,NHL,2018,Pittsburgh,0.618421


In [11]:
## MERGE ALL SPORTS INTO SINGLE DATAFRAME
all_sports = (cities[['Metropolitan area', 'Population']]
              .merge(final_df_nhl[['Metropolitan area', 'AvgWL_Ratio_nhl']], how = 'outer', on='Metropolitan area')
              .merge(final_df_nba[['Metropolitan area', 'AvgWL_Ratio_nba']], how = 'outer', on='Metropolitan area')
              .merge(final_df_mlb[['Metropolitan area', 'AvgWL_Ratio_mlb']], how = 'outer', on='Metropolitan area')
              .merge(final_df_nfl[['Metropolitan area', 'AvgWL_Ratio_nfl']], how = 'outer', on='Metropolitan area'))
all_sports

,Metropolitan area,Population,AvgWL_Ratio_nhl,AvgWL_Ratio_nba,AvgWL_Ratio_mlb,AvgWL_Ratio_nfl
0,New York City,20153634,0.518201,0.347561,0.546296,0.281250
1,Los Angeles,13310447,0.622895,0.469512,0.529122,0.781250
2,San Francisco Bay Area,6657982,0.625000,0.707317,0.524691,0.250000
3,Chicago,9512999,0.458333,0.329268,0.482769,0.750000
4,Dallas–Fort Worth,7233323,0.567568,0.292683,0.413580,0.625000
5,"Washington, D.C.",6131977,0.653333,0.524390,0.506173,0.437500
6,Philadelphia,6070500,0.617647,0.634146,0.493827,0.562500
7,Boston,4794447,0.714286,0.670732,0.666667,0.687500
8,Minneapolis–Saint Paul,3551036,0.633803,0.573171,0.481481,0.533333
9,Denver,2853077,0.589041,0.560976,0.558282,0.375000


### **Questions 1-4 Answer**: 
For each sport league, calculate the correlation (Pearson R) between Win-Loss Ratio average and Population. Each result should only include locales with teams representing the league (e.g. Do not include Greenay in NHL calculation)

In [12]:
answers = {}
for sport in sports:
    avgRatioTemp = 'AvgWL_Ratio_'+sport
    subsetDfTemp = all_sports[np.isnan(all_sports[avgRatioTemp])==False]
    
    population_by_region = subsetDfTemp['Population'].astype('int64').tolist() # pass in metropolitan area population from cities
    win_loss_by_region = subsetDfTemp[avgRatioTemp].tolist() # pass in win/loss ratio from nhl_df in the same order as cities["Metropolitan area"]

    answers[sport] = stats.pearsonr(population_by_region, win_loss_by_region)[0]

answers

{'nhl': 0.012486162921209912,
 'nba': -0.17657160252844617,
 'mlb': 0.1502769830266931,
 'nfl': 0.004922112149349386}

***Q 1-4 Analysis:*** this experiment aimed to evaluate the relationship between sport team performance (measured as *Average Win-Loss Ratio*) and *Population* of the corresponding metropolitan area, sampling all teams from the four (4) largest major sports leagues {NHL, NBA, NFL, MLB}. Correlation strength was tested using the Pearson Correlation Coefficient.

Two (2) leagues demonstrated a statistically significant correlation {p_NFL=0.005 and p_NHL=0.012} while two (2) fell below the threshold of significance {p_NBA= -0.177 and p_MLB=0.150}. Three (3) tests {NHL, MLB, NFL} indicated a postive correlation betwen performance and population, while one (1) {NBA} indicated a negative correlation.

With only 50% of the leagues indicating a statistically significant correlation, we can conclude that this evidence is not sufficient to support the hypothesis. Nevertheless, the strength of some of these results warrants additional testing--in particular the NBA (p_NBA=0.005).

My recommendations for additonal experimentation are the following: (1) utilizing the same experiment design, increase the years represented by the data and (2) control for both individual team and city resources.

## Question 5
In this question I would like you to explore the hypothesis that **given that an area has two sports teams in different sports, those teams will perform the same within their respective sports**. How I would like to see this explored is with a series of paired t-tests (so use [`ttest_rel`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_rel.html)) between all pairs of sports. Are there any sports where we can reject the null hypothesis? Again, average values where a sport has multiple teams in one region. Remember, you will only be including, for each sport, cities which have teams engaged in that sport, drop others as appropriate. This question is worth 20% of the grade for this assignment.

In [13]:
p_values = pd.DataFrame({k:np.nan for k in sports}, index=sports)
p_values

,nhl,nba,mlb,nfl
nhl,NaN,NaN,NaN,NaN
nba,NaN,NaN,NaN,NaN
mlb,NaN,NaN,NaN,NaN
nfl,NaN,NaN,NaN,NaN


In [14]:
## (1) nhl+nba (2) nhl+mlb (3) nhl+nfl (4) nba+mlb (5) nba+nfl (6) mlb+nfl
    
for sport1 in sports:
    for sport2 in sports:
        print(sport1, sport2)
        tmpVal = np.nan

        if sport1==sport2:
            p_values.loc[sport1,sport2] = tmpVal
            print('test size: NA')
        else:
            col1='AvgWL_Ratio_'+sport1.lower()
            col2='AvgWL_Ratio_'+sport2.lower()
            tmpDF = all_sports[(np.isnan(all_sports[col1])==False) & (np.isnan(all_sports[col2])==False)]

            print('test size: ',len(tmpDF))
            
            tmpVal = stats.ttest_rel(tmpDF[col1], tmpDF[col2])[1]
            p_values.loc[sport1,sport2] = tmpVal

p_values

nhl nhl
test size: NA
nhl nba
test size:  14
nhl mlb
test size:  17
nhl nfl
test size:  17
nba nhl
test size:  14
nba nba
test size: NA
nba mlb
test size:  18
nba nfl
test size:  19
mlb nhl
test size:  17
mlb nba
test size:  18
mlb mlb
test size: NA
mlb nfl
test size:  22
nfl nhl
test size:  17
nfl nba
test size:  19
nfl mlb
test size:  22
nfl nfl
test size: NA


,nhl,nba,mlb,nfl
nhl,NaN,0.022297,0.000708,0.030883
nba,0.022297,NaN,0.950540,0.941792
mlb,0.000708,0.950540,NaN,0.802069
nfl,0.030883,0.941792,0.802069,NaN


***Q5 Analysis:*** This experiment aims to test the hypothesis that teams within the same metropolitan area, but from different leagues, perform similarly where performance is measured by *Average Win-Loss Ration*.

All teams from the four (4) largest major sports league {NHL, NBA, MLB, NFL} were compared by T-Test in all possibile permutations,of which there are 6 {NHL-NBA, NHL-MLB, NHL-NFL, NBA-MLB, NBA-NFL, MLB-NFL}, but excluding a test against itself (e.g.NHL-NHL). 

Of all test permutations, only those including the NHL indicated a statistically significant result. As such, we could say that sport leagues in locales with an NHL team *may* tend to perform as well within their respective leagues. However, being that this is the only test producing statistically significant results (p<0.05), we can conclude that there is not sufficient evidence to globally reject the Null Hypothesis.

Furthermore, it should be emphasized that this experiment only included data from 2018. To further explore the hypothesis, I would make the following recommendations: (1) include additional years of data; (2) being that tests including the NHL had the smallest sample size at 17, investigate the potential impact of sample size on results; and (3) expand the scope of the experiment by including additional controls for factors that may contribute to league performance such as  differences in monetary and non-monetary support in a given metropolitan area.